# Historico NOAA Temperatura en Aeropuertos Brasil

In [0]:
# Databricks Notebook - NOAA Historical Backfill

import gzip
import json
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import requests

BASE_URL = "https://www.ncei.noaa.gov/pub/data/noaa/isd-lite"

VOLUME_BASE = Path("/Volumes/weather/raw/noaa_volume")
GZ_DIR = VOLUME_BASE / "gz/historical"
JSON_DIR = VOLUME_BASE / "json/historical"

GZ_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)


@dataclass(frozen=True)
class StationMeta:
    icao: str
    isd: str
    name: str


STATIONS = {
    "SBGR": StationMeta("SBGR", "830750-99999", "São Paulo/Guarulhos Intl, SP, BR"),
    "SBGL": StationMeta("SBGL", "837460-99999", "Rio de Janeiro/Galeão Intl, RJ, BR"),
    "SBPA": StationMeta("SBPA", "839710-99999", "Porto Alegre/Salgado Filho, RS, BR"),
    "SBFL": StationMeta("SBFL", "838990-99999", "Florianopolis Intl, SC, BR"),
}

In [0]:
def iter_years(start_year: int, end_year: int):
    if end_year < start_year:
        raise ValueError("end_year must be >= start_year")
    return range(start_year, end_year + 1)


def build_url(meta: StationMeta, year: int):
    return f"{BASE_URL}/{year}/{meta.isd}-{year}.gz"

In [0]:
def download_if_needed(meta: StationMeta, year: int, session: requests.Session):
    local_path = GZ_DIR / f"{meta.icao}_{year}.gz"

    if local_path.exists():
        print(f"✓ {local_path.name} ya existe")
        return local_path

    url = build_url(meta, year)
    print(f"↓ Descargando {url}")

    try:
        resp = session.get(url, timeout=120)
        if resp.status_code == 404:
            print(f"  → No existe en NOAA (probable sin datos)")
            return None
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"  ✗ Error: {e}")
        return None

    local_path.write_bytes(resp.content)
    print(f"  ✓ Guardado {local_path.name}")
    return local_path

In [0]:
def parse_isd_line(line: str):
    parts = line.split()
    if len(parts) < 11:
        return None

    year, month, day, hour = map(int, parts[:4])
    air_temp, dewpt, _, wind_dir, wind_spd, _, visibility, slp = parts[4:12]

    def parse_value(v, scale=None):
        if v == "-9999":
            return None
        val = float(v)
        return val / scale if scale else val

    dt = datetime(year, month, day, hour, tzinfo=timezone.utc)

    return {
        "obsTime": int(dt.timestamp()),
        "reportTime": dt.isoformat().replace("+00:00", "Z"),
        "temp": parse_value(air_temp, 10),
        "dewp": parse_value(dewpt, 10),
        "wdir": parse_value(wind_dir),
        "wspd": parse_value(wind_spd, 10),
        "visib": parse_value(visibility),
        "altim": parse_value(slp, 10),
    }

In [0]:
def transform_if_needed(meta: StationMeta, year: int):
    gz_path = GZ_DIR / f"{meta.icao}_{year}.gz"
    json_path = JSON_DIR / f"{meta.icao}_{year}.json"

    if not gz_path.exists():
        return

    if json_path.exists():
        print(f"✓ JSON ya existe {json_path.name}")
        return

    print(f"→ Transformando {gz_path.name}")

    records = []

    with gzip.open(gz_path, "rt", encoding="utf-8") as f:
        for line in f:
            parsed = parse_isd_line(line)
            if parsed:
                parsed["icaoId"] = meta.icao
                parsed["name"] = meta.name
                records.append(parsed)

    json_path.write_text(
        json.dumps(records, ensure_ascii=False),
        encoding="utf-8"
    )

    print(f"  ✓ {len(records)} registros -> {json_path.name}")

In [0]:
START_YEAR = 1990
END_YEAR = 2025

session = requests.Session()

for meta in STATIONS.values():
    print(f"\n=== {meta.icao} ===")
    for year in iter_years(START_YEAR, END_YEAR):
        gz_file = download_if_needed(meta, year, session)
        if gz_file:
            transform_if_needed(meta, year)

# Transformacion de los ultimos meses del año dado que no estna ten el historial ni en la api

In [0]:
import json
import pandas as pd
from pathlib import Path
import numpy as np

CSV_PATH = "/Volumes/weather/raw/noaa_volume/gz/asos.csv"
OUTPUT_DIR = Path("/Volumes/weather/raw/noaa_volume/json/daily")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH, comment="#")

# Parse fecha UTC
df["valid"] = pd.to_datetime(df["valid"], utc=True)

# Conversiones
df["temp"] = (df["tmpf"] - 32) * 5/9
df["dewp"] = (df["dwpf"] - 32) * 5/9
df["altim"] = df["alti"] * 33.8639  # inHg → hPa

# Epoch
df["obsTime"] = df["valid"].astype("int64") // 10**9

# ISO reportTime
df["reportTime"] = df["valid"].dt.strftime("%Y-%m-%dT%H:%M:%S.000Z")

df["icaoId"] = df["station"]
df["rawOb"] = df["metar"]
df["wdir"] = df["drct"]
df["wspd"] = df["sknt"]
df["visib"] = df["vsby"]

# Columnas finales (alineadas con bronze mínimo)
final_cols = [
    "icaoId",
    "obsTime",
    "reportTime",
    "temp",
    "dewp",
    "wdir",
    "wspd",
    "visib",
    "altim",
    "lat",
    "lon",
    "rawOb"
]

df = df[final_cols]

# Reemplazar NaN por None
df = df.replace({np.nan: None})

df["date"] = pd.to_datetime(df["reportTime"]).dt.date

grouped = df.groupby("date")

print(f"Días encontrados: {len(grouped)}")

for day, group in grouped:
    records = group.drop(columns=["date"]).to_dict(orient="records")
    file_path = OUTPUT_DIR / f"METAR_{day:%Y_%m_%d}.json"

    with file_path.open("w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False)

    print(f"✓ {file_path.name} ({len(records)} registros)")

print("Transformación finalizada.")

In [0]:
HIST_PATH = "/Volumes/weather/raw/noaa_volume/json/historical/"

# Leer todos los json de manera recursiva
historical_df = (
    spark.read
         .option("multiLine", True)
         .option("recursiveFileLookup", "true")
         .json(HIST_PATH)
)

total_records = historical_df.count()
total_files = len(historical_df.inputFiles())

print(f"Archivos leídos: {total_files}")
print(f"Total registros históricos: {total_records}")

In [0]:
historical_df.groupBy("icaoId").count().orderBy("icaoId").display()

In [0]:
from pyspark.sql.functions import min, max

historical_df.select(
    min("reportTime"),
    max("reportTime")
).display()

In [0]:
from pyspark.sql.functions import min, max

(
    historical_df
        .groupBy("icaoId")
        .agg(
            min("reportTime").alias("min_reportTime"),
            max("reportTime").alias("max_reportTime")
        )
        .orderBy("icaoId")
        .display()
)


#